# 03. Modelos Bayesianos Discretos, Clasificación de Texto (NLP) y Datos Categóricos

**Módulo:** `05_Naive_Bayes`  
**Objetivo:** Dominar la taxonomía matemática y computacional de los clasificadores Naive Bayes para datos discretos (`MultinomialNB`, `BernoulliNB`, `CategoricalNB` y `ComplementNB`), deducir el Suavizado de Laplace y Lidstone para resolver el problema de la frecuencia cero, demostrar cómo `BernoulliNB` penaliza explícitamente la ausencia de características, manejar variables tabulares de distinta cardinalidad y construir un Pipeline de Procesamiento de Lenguaje Natural (NLP) de grado industrial optimizado con `GridSearchCV`.

---

## 1. Taxonomía de Clasificadores Bayesianos Discretos

Cuando nuestras características $\mathbf{x} = (x_1, x_2, \dots, x_D)$ representan conteos enteros, indicadores binarios de presencia/ausencia o códigos de categorías nominales, asumir una campana Gaussiana continua es conceptualmente incorrecto. Scikit-Learn provee cuatro variantes especializadas según la distribución generativa de $P(\mathbf{X} \mid Y = c_k)$:

| Clasificador en Scikit-Learn | Distribución Generativa Subyacente | Dominio de $x_j$ | Caso de Uso Principal en la Industria |
| :--- | :--- | :--- | :--- |
| **`MultinomialNB`** | Multinomial sobre conteos de tokens (o pesos TF-IDF) | $x_j \in \mathbb{N}_0 \; (\text{o } \mathbb{R}_{\ge 0})$ | Clasificación de documentos largos/medianos (*Bag-of-Words*, TF-IDF). |
| **`BernoulliNB`** | $D$ ensayos de Bernoulli independientes por característica | $x_j \in \{0, 1\}$ | Textos cortos (SMS, tweets, titulares) donde la **ausencia** de una palabra es informativa, o vectores binarios (*One-Hot*). |
| **`CategoricalNB`** | Distribución Categórica (Multinoulli) independiente por columna | $x_j \in \{0, 1, \dots, K_j - 1\}$ | Datos tabulares con variables categóricas nominales/ordinales codificadas con `OrdinalEncoder`. |
| **`ComplementNB`** | Multinomial estimada sobre el **complemento** de la clase ($\tilde{c}_k$) | $x_j \in \mathbb{N}_0 \; (\text{o } \mathbb{R}_{\ge 0})$ | Clasificación de texto con **severo desbalance de clases** (Rennie et al., 2003). |

In [ ]:
# Configuración del entorno y librerías estándar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

from sklearn.naive_bayes import MultinomialNB, BernoulliNB, CategoricalNB, ComplementNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# Estándar visual del repositorio
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)

print("Entorno configurado correctamente.")

---

## 2. El Problema de la Frecuencia Cero y Deducción del Suavizado de Laplace / Lidstone

### Formulación del Modelo Multinomial
En clasificación de texto bajo representación *Bag-of-Words*, un documento se representa mediante un vector $\mathbf{x} = (x_1, x_2, \dots, x_V) \in \mathbb{N}_0^V$, donde $V$ es el tamaño del vocabulario y $x_j$ es el número de veces que la palabra $w_j$ aparece en el documento.  
Bajo la distribución Multinomial condicionada a la clase $c_k$, la probabilidad de observar el documento $\mathbf{x}$ de longitud total $L = \sum_{j=1}^V x_j$ es:

$$P(\mathbf{x} \mid Y = c_k) = \frac{L!}{\prod_{j=1}^{V} x_j!} \prod_{j=1}^{V} \theta_{kj}^{x_j}$$

donde $\theta_{kj} = P(w_j \mid Y = c_k)$ es la probabilidad de que un token generado por la clase $c_k$ sea la palabra $w_j$ (con $\sum_{j=1}^V \theta_{kj} = 1$). Al aplicar la regla MAP en espacio logarítmico, el coeficiente multinomial factorial $\frac{L!}{\prod x_j!}$ no depende de la clase $c_k$ y se descarta, resultando en un clasificador **lineal respecto a los conteos $x_j$**:

$$\hat{y}_{\text{MAP}} = \underset{c_k \in \mathcal{Y}}{\arg\max} \; \left[ \ln P(Y = c_k) + \sum_{j=1}^{V} x_j \cdot \ln \theta_{kj} \right]$$

### La Catástrofe de la Frecuencia Cero en MLE
El estimador de Máxima Verosimilitud (MLE) para $\theta_{kj}$ divide el conteo total de la palabra $w_j$ en todos los documentos de la clase $c_k$ ($N_{kj}$) entre el total de palabras sumadas en la clase $c_k$ ($N_k = \sum_{m=1}^V N_{km}$):

$$\hat{\theta}_{kj}^{\text{MLE}} = \frac{N_{kj}}{N_k}$$

**¿Qué sucede si en producción aparece una palabra $w_{j^*}$ que jamás se vio en los ejemplos de entrenamiento de la clase $c_k$ ($N_{kj^*} = 0$)?**
* Entonces $\hat{\theta}_{kj^*}^{\text{MLE}} = 0$.
* En el producto de probabilidades: $P(\mathbf{x} \mid Y = c_k) = \prod_{j=1}^V \theta_{kj}^{x_j} = 0$ (una sola palabra nueva aniquila la evidencia de todas las demás palabras del documento).
* En el espacio logarítmico: $\ln(0) = -\infty$.

### Solución Bayesiana (Prior de Dirichlet): Suavizado de Laplace y Lidstone
Desde la estadística bayesiana, si asignamos una distribución a priori conjugada **Dirichlet simétrica** $\text{Dir}(\alpha, \alpha, \dots, \alpha)$ sobre el vector de parámetros $\boldsymbol{\theta}_k$, el estimador MAP añade $\alpha$ "pseudoconteos" virtuales a cada palabra del vocabulario $V$:

$$\hat{\theta}_{kj}^{(\alpha)} = \frac{N_{kj} + \alpha}{N_k + \alpha \cdot V}$$

Observa que la suma de probabilidades sobre todo el vocabulario sigue siendo exactamente 1:

$$\sum_{j=1}^{V} \hat{\theta}_{kj}^{(\alpha)} = \frac{\sum_{j=1}^{V} N_{kj} + \sum_{j=1}^{V} \alpha}{N_k + \alpha V} = \frac{N_k + \alpha V}{N_k + \alpha V} = 1$$

* **Suavizado de Laplace (*Add-One Smoothing*):** Caso $\alpha = 1.0$.
* **Suavizado de Lidstone:** Caso general $0 < \alpha < 1$ (muy común en NLP con TF-IDF, ej. $\alpha = 0.1$ o $\alpha = 0.01$, para no diluir excesivamente los conteos en vocabularios de decenas de miles de términos).
* **Límite cuando $\alpha \to \infty$:** Todas las verosimilitudes convergen a la distribución uniforme $1/V$, y el clasificador predice únicamente usando el *Prior* $P(Y = c_k)$.

In [ ]:
# Demostración Matemática y Visual del Efecto de alpha (Suavizado de Lidstone/Laplace)
vocabulario = ['oferta', 'gratis', 'urgente', 'reunion', 'proyecto', 'informe', 'cripto_nuevo']
V = len(vocabulario)

# Conteos observados N_{k,j} en la clase 'Spam' (nota que 'informe' y 'cripto_nuevo' tienen conteo 0)
conteos_spam = np.array([45, 38, 25, 2, 1, 0, 0], dtype=float)
N_spam = np.sum(conteos_spam)

alphas = [0.0, 0.1, 1.0, 10.0, 100.0]
resultados_theta = {}

for a in alphas:
    theta_a = (conteos_spam + a) / (N_spam + a * V)
    resultados_theta[f"alpha = {a}"] = theta_a

df_smoothing = pd.DataFrame(resultados_theta, index=vocabulario)
print("=== TABLA DE PROBABILIDADES ESTIMADAS P(w_j | Y=Spam) SEGÚN ALPHA ===")
display(df_smoothing.style.format("{:.5f}"))

# Gráfico de transición desde MLE (alpha=0) hacia la Distribución Uniforme (1/V)
fig, ax = plt.subplots(figsize=(11, 5))
x_pos = np.arange(V)
width = 0.18
colores_alpha = ['#d62728', '#ff7f0e', '#1f77b4', '#2ca02c', '#9467bd']

for idx, col in enumerate(df_smoothing.columns):
    ax.bar(x_pos + (idx - 2)*width, df_smoothing[col], width=width, label=col, color=colores_alpha[idx], alpha=0.88)

ax.axhline(1.0 / V, color='black', linestyle='--', lw=1.5, label=f'Distribución Uniforme (1/V = {1/V:.3f})')
ax.set_xticks(x_pos)
ax.set_xticklabels(vocabulario, fontsize=11, fontweight='bold')
ax.set_ylabel(r'Probabilidad Estimada $\hat{\theta}_{kj}^{(\alpha)}$', fontsize=11)
ax.set_title(r'Efecto Regularizador del Hiperparámetro $\alpha$: De Frecuencia Cero a Prior Uniforme', fontsize=12.5, fontweight='bold')
ax.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()

---

## 3. `BernoulliNB` vs. `MultinomialNB`: El Poder de Penalizar la Ausencia

Una distinción crucial en ingeniería de características para texto y datos binarios radica en cómo `MultinomialNB` y `BernoulliNB` tratan las características cuyo valor es cero ($x_j = 0$):

### 1. En `MultinomialNB` (Ignora las ausencias):
La log-verosimilitud es $\sum_{j=1}^{V} x_j \ln \theta_{kj}$. Si una palabra $w_j$ **no aparece** en el documento ($x_j = 0$), su término en la suma es $0 \cdot \ln \theta_{kj} = 0$. Es decir, **`MultinomialNB` solo acumula evidencia de las palabras presentes**.

### 2. En `BernoulliNB` (Penaliza explícitamente las ausencias):
`BernoulliNB` modela cada palabra del vocabulario como una variable binaria $b_j = \mathbb{I}(x_j > \text{binarize}) \in \{0, 1\}$. La probabilidad de observar el vector binario $\mathbf{b} \in \{0, 1\}^V$ en la clase $c_k$ es:

$$P(\mathbf{b} \mid Y = c_k) = \prod_{j=1}^{V} p_{kj}^{b_j} \left(1 - p_{kj}\right)^{1 - b_j}$$

donde $p_{kj} = P(b_j = 1 \mid Y = c_k)$ es la fracción de documentos de la clase $c_k$ que contienen al menos una vez el término $w_j$. Aplicando logaritmos y reordenando algebraicamente:

$$\ln P(\mathbf{b} \mid Y = c_k) = \sum_{j=1}^{V} \left[ b_j \ln p_{kj} + (1 - b_j) \ln(1 - p_{kj}) \right] = \sum_{j=1}^{V} b_j \ln \frac{p_{kj}}{1 - p_{kj}} + \underbrace{\sum_{j=1}^{V} \ln(1 - p_{kj})}_{\text{Penalización base por no aparición}}$$

> **Intuición Clave:** Si la palabra `"estimado"` aparece en el 95% de los correos corporativos formales ($p_{kj} = 0.95$) y llega un mensaje corto que **NO** contiene la palabra `"estimado"` ($b_j = 0$), `BernoulliNB` multiplica la probabilidad de esa clase por $(1 - 0.95) = 0.05$ (suma $\ln(0.05) = -3.0$), castigando fuertemente a la clase formal por la ausencia del término esperado.

In [ ]:
# Demostración Experimental: Cómo BernoulliNB penaliza la ausencia de palabras clave
corpus_mini = [
    "estimado director adjunto informe trimestral finanzas",
    "estimado equipo adjunto minuta reunion proyecto",
    "estimado cliente adjunto factura contrato anual",
    "estimado comité adjunto balance auditoria",
    "gana dinero rapido casino apuesta bono",
    "bono gratis casino ruleta gana premio",
    "apuesta rapido gana premio efectivo ya",
    "casino gratis bono efectivo rapido"
]
# 0 = Corporativo Formal (siempre incluye 'estimado' y 'adjunto'), 1 = Spam
y_mini = np.array([0, 0, 0, 0, 1, 1, 1, 1])

vec_count = CountVectorizer()
X_mini = vec_count.fit_transform(corpus_mini)

mnb_mini = MultinomialNB(alpha=0.5).fit(X_mini, y_mini)
bnb_mini = BernoulliNB(alpha=0.5, binarize=0.0).fit(X_mini, y_mini)

# Evaluamos un mensaje neutral muy corto donde NO aparecen las palabras obligatorias ('estimado', 'adjunto')
doc_prueba = ["reunion mañana"]
X_prueba = vec_count.transform(doc_prueba)

prob_mnb = mnb_mini.predict_proba(X_prueba)[0]
prob_bnb = bnb_mini.predict_proba(X_prueba)[0]

print(f"Documento evaluado: {doc_prueba[0]!r} (Nota: Ausencia de 'estimado' y 'adjunto')\n")
print(f"MultinomialNB (Ignora ausencias)    -> P(Formal) = {prob_mnb[0]:.4f} | P(Spam) = {prob_mnb[1]:.4f}")
print(f"BernoulliNB   (Penaliza ausencias)  -> P(Formal) = {prob_bnb[0]:.4f} | P(Spam) = {prob_bnb[1]:.4f}\n")

# Inspeccionamos la probabilidad aprendida p_{k,j} en BernoulliNB para 'estimado' y 'adjunto'
vocab_dict = vec_count.vocabulary_
for palabra in ['estimado', 'adjunto', 'reunion']:
    idx_w = vocab_dict[palabra]
    p_formal = np.exp(bnb_mini.feature_log_prob_[0, idx_w])
    p_spam = np.exp(bnb_mini.feature_log_prob_[1, idx_w])
    print(f"Palabra '{palabra:<9}' -> P(b_j=1 | Formal) = {p_formal:.3f} (Castigo si falta: 1-p = {1-p_formal:.3f}) | P(b_j=1 | Spam) = {p_spam:.3f}")

---

## 4. `CategoricalNB` para Datos Tabulares y el Rol Crítico de `min_categories`

Cuando trabajamos con tablas de bases de datos relacionales que contienen atributos categóricos discretos (ej. `Tipo_Contrato`, `Metodo_Pago`, `Segmento_Cliente`), **cometer el error de aplicar `OneHotEncoder` seguido de `GaussianNB` o `MultinomialNB` viola el modelo generativo**:
* Si una variable tiene 5 categorías mutuamente excluyentes y creamos 5 columnas binarias *One-Hot*, `BernoulliNB` asume que esas 5 columnas son independientes entre sí (¡como si un cliente pudiera tener 3 estados civiles simultáneamente o ninguno!).
* **`CategoricalNB`** modela correctamente cada columna original $j$ (codificada con enteros $0, 1, \dots, K_j - 1$ vía `OrdinalEncoder`) mediante una distribución Categórica propia donde las probabilidades de sus $K_j$ niveles suman 1.

### El Peligro en Producción de Categorías Raras y `min_categories`
Si durante la validación cruzada (`GridSearchCV`) una categoría rara (ej. el nivel `4` en una columna con valores `0, 1, 2, 3, 4`) cae exclusivamente en el conjunto de *Test* y no aparece en *Train*, `CategoricalNB` inferirá que esa característica solo tiene $K_j = 4$ niveles. Al predecir sobre *Test*, lanzará un `IndexError: index 4 is out of bounds`.  
Para blindar el modelo en producción, debemos informar explícitamente la cardinalidad conocida de cada columna mediante el parámetro **`min_categories`**.

In [ ]:
# Simulación Financiera/Demográfica Tabular con CategoricalNB y Protección con min_categories
np.random.seed(42)
n_clientes = 800

# Cardinalidades reales conocidas del negocio:
# Col 0: Nivel Educativo (4 niveles: 0..3)
# Col 1: Tipo de Vivienda (3 niveles: 0..2)
# Col 2: Región Geográfica (6 niveles: 0..5, donde el nivel 5 es una región insular muy rara)
cardinalidades_reales = [4, 3, 6]

# Generamos datos de entrenamiento donde la región 5 NO apareció por azar muestral
X_tab_train = np.column_stack([
    np.random.choice(4, size=n_clientes, p=[0.2, 0.5, 0.2, 0.1]),
    np.random.choice(3, size=n_clientes, p=[0.4, 0.4, 0.2]),
    np.random.choice(5, size=n_clientes, p=[0.3, 0.3, 0.2, 0.15, 0.05]) # Solo niveles 0..4 en Train
])
# Etiqueta de incumplimiento (Default) dependiente de las variables
logits = -1.2 + 0.6*(X_tab_train[:, 0] == 0) + 0.8*(X_tab_train[:, 1] == 2) + 0.5*(X_tab_train[:, 2] >= 3)
probs_default = 1 / (1 + np.exp(-logits))
y_tab_train = np.random.binomial(1, probs_default)

# Llega un nuevo cliente en Test perteneciente a la Región 5 (categoría no vista en Train)
X_tab_test_raro = np.array([[1, 2, 5]])

# 1. Sin min_categories -> Falla en producción al ver el índice 5
cat_nb_fragil = CategoricalNB(alpha=1.0).fit(X_tab_train, y_tab_train)
try:
    cat_nb_fragil.predict_proba(X_tab_test_raro)
except IndexError as e:
    print(f"❌ Error capturado en CategoricalNB sin min_categories: {e}")

# 2. Con min_categories -> Blindado mediante Suavizado de Laplace en la categoría 5
cat_nb_robusto = CategoricalNB(alpha=1.0, min_categories=cardinalidades_reales).fit(X_tab_train, y_tab_train)
prob_raro = cat_nb_robusto.predict_proba(X_tab_test_raro)[0]
print(f"✅ CategoricalNB con min_categories={cardinalidades_reales} predice sin errores:")
print(f"   P(Paga=0 | X) = {prob_raro[0]:.4f} | P(Default=1 | X) = {prob_raro[1]:.4f}")
print(f"   Dimensiones de la tabla de log-probabilidad de Región (Col 2): {cat_nb_robusto.feature_log_prob_[2].shape}")

---

## 5. `ComplementNB` para Desbalance de Clases Severo

En problemas reales de filtrado de spam, detección de discurso de odio o clasificación de tickets de soporte críticos, el dataset suele estar **fuertemente desbalanceado** (ej. 92% clase normal vs. 8% clase minoritaria). En este escenario, `MultinomialNB` sufre dos sesgos graves identificados por **Rennie et al. (ICML 2003)**:
1. Los conteos $N_{kj}$ de la clase mayoritaria están estimados sobre miles de documentos, mientras que los de la clase minoritaria tienen alta varianza muestral.
2. Los pesos de verosimilitud $\ln \hat{\theta}_{kj}$ de la clase con más documentos dominan sistemáticamente el producto punto.

### La Formulación Matemática de `ComplementNB` (CNB)
En lugar de estimar la probabilidad de que una palabra aparezca en la clase $c_k$, **`ComplementNB` estima la probabilidad de que la palabra aparezca en todas las demás clases EXCEPTO $c_k$** (el complemento $\tilde{c}_k = \mathcal{Y} \setminus \{c_k\}$):

$$\hat{\theta}_{\tilde{k}j} = \frac{\left(\sum_{c_m \neq c_k} N_{mj}\right) + \alpha}{\left(\sum_{c_m \neq c_k} N_m\right) + \alpha V}$$

Posteriormente, toma los pesos negativos $w_{kj} = -\ln \hat{\theta}_{\tilde{k}j}$ (opcionalmente normalizados en norma $L_1$: $w_{kj} \leftarrow w_{kj} / \sum_m |w_{km}|$) y asigna la clase que **menos se ajusta a su complemento** (es decir, minimiza la coincidencia con las clases opuestas):

$$\hat{y}_{\text{CNB}} = \underset{c_k \in \mathcal{Y}}{\arg\min} \; \sum_{j=1}^{V} x_j \cdot w_{kj}$$

**¿Por qué funciona tan bien en clases minoritarias?** Porque para estimar los parámetros de la clase minoritaria, utiliza la enorme cantidad de datos de todas las clases mayoritarias combinadas, logrando estimaciones de varianza ultra baja y eliminando el sesgo del tamaño de muestra.

---

## 6. Proyecto Real de NLP: Clasificación de Documentos Técnicos y Detección bajo Desbalance

A continuación, construiremos un banco de pruebas de Procesamiento de Lenguaje Natural (NLP) completamente reproducible y auto-contenido (con estructura homóloga a `fetch_20newsgroups` en 4 categorías técnicas: `sci.crypt`, `comp.graphics`, `rec.autos` y `med.clinical`), evaluando:
1. La superioridad de **`ComplementNB` frente a `MultinomialNB`** bajo desbalance severo.
2. Un **Pipeline completo con `CountVectorizer` vs `TfidfVectorizer` y `GridSearchCV`**.
3. **Matriz de Confusión** e **Interpretabilidad de Términos con Mayor Poder Discriminativo**.

In [ ]:
# Construcción de Corpus Realista Multi-Categoría (Estilo 20 Newsgroups) con vocabulario compartido y ruido
np.random.seed(42)

categorias_nlp = ['comp.graphics', 'sci.crypt', 'rec.autos', 'sci.med']

vocab_especifico = {
    'comp.graphics': [
        'opengl', 'shader', 'polygon', 'texture', 'rendering', 'raytracing', 'gpu', 'pixel',
        'vertex', 'framebuffer', 'aliasing', 'mesh', 'vector', 'vulkan', '3d', 'animation',
        'bitmap', 'resolution', 'color', 'display', 'matrix', 'transform', 'lighting', 'scene'
    ],
    'sci.crypt': [
        'cipher', 'encryption', 'rsa', 'public_key', 'hash', 'signature', 'cryptography', 'nsa',
        'algorithm', 'prime', 'aes', 'diffie_hellman', 'plaintext', 'decrypt', 'key_exchange', 'entropy',
        'block', 'protocol', 'security', 'authentication', 'bit', 'random', 'attack', 'certificate'
    ],
    'rec.autos': [
        'engine', 'transmission', 'torque', 'horsepower', 'brake', 'suspension', 'turbo', 'sedan',
        'clutch', 'exhaust', 'tire', 'fuel', 'cylinder', 'gearbox', 'chassis', 'steering',
        'highway', 'dealer', 'oil', 'mileage', 'radiator', 'spark_plug', 'valve', 'diesel'
    ],
    'sci.med': [
        'patient', 'diagnosis', 'symptom', 'treatment', 'clinical', 'therapy', 'vaccine', 'antibiotic',
        'infection', 'syndrome', 'dosage', 'chronic', 'biopsy', 'surgery', 'placebo', 'pathology',
        'immune', 'Oncology', 'cardiac', 'blood', 'physician', 'trial', 'hospital', 'disease'
    ]
}

vocab_comun = [
    'system', 'problem', 'question', 'article', 'university', 'research', 'data', 'analysis',
    'performance', 'standard', 'report', 'version', 'method', 'result', 'group', 'information',
    'test', 'model', 'time', 'high', 'low', 'case', 'issue', 'update', 'source', 'paper', 'user'
]

def generar_corpus_newsgroups(n_por_clase: list, seed: int = 42):
    rng = np.random.default_rng(seed)
    textos, etiquetas = [], []
    for c_idx, (cat, n_docs) in enumerate(zip(categorias_nlp, n_por_clase)):
        palabras_cat = vocab_especifico[cat]
        otras_cats = [k for k in categorias_nlp if k != cat]
        for _ in range(n_docs):
            long_doc = rng.integers(18, 45)
            # Mezcla realista: 45% palabras de la categoría, 40% vocabulario común, 15% ruido cruzado de otras clases
            n_prop = int(long_doc * 0.45)
            n_com = int(long_doc * 0.40)
            n_ruido = long_doc - n_prop - n_com
            
            tokens_prop = rng.choice(palabras_cat, size=n_prop, replace=True)
            tokens_com = rng.choice(vocab_comun, size=n_com, replace=True)
            cat_ruido = rng.choice(otras_cats)
            tokens_ruido = rng.choice(vocab_especifico[cat_ruido], size=n_ruido, replace=True)
            
            doc_tokens = np.concatenate([tokens_prop, tokens_com, tokens_ruido])
            rng.shuffle(doc_tokens)
            textos.append(" ".join(doc_tokens))
            etiquetas.append(c_idx)
    return textos, np.array(etiquetas)

# 1. Experimento con Desbalance Severo (Clases mayoritarias vs Clase minoritaria 'sci.med' al 5%)
textos_imb, y_imb = generar_corpus_newsgroups([600, 600, 500, 90], seed=42)
X_tr_imb, X_te_imb, y_tr_imb, y_te_imb = train_test_split(
    textos_imb, y_imb, test_size=0.3, stratify=y_imb, random_state=42
)

tfidf_imb = TfidfVectorizer(ngram_range=(1, 2))
X_tr_vec = tfidf_imb.fit_transform(X_tr_imb)
X_te_vec = tfidf_imb.transform(X_te_imb)

mnb_imb = MultinomialNB(alpha=1.0).fit(X_tr_vec, y_tr_imb)
cnb_imb = ComplementNB(alpha=1.0).fit(X_tr_vec, y_tr_imb)

f1_macro_mnb = f1_score(y_te_imb, mnb_imb.predict(X_te_vec), average='macro')
f1_macro_cnb = f1_score(y_te_imb, cnb_imb.predict(X_te_vec), average='macro')

f1_minoritaria_mnb = f1_score(y_te_imb == 3, mnb_imb.predict(X_te_vec) == 3)
f1_minoritaria_cnb = f1_score(y_te_imb == 3, cnb_imb.predict(X_te_vec) == 3)

print("=== COMPARATIVA EN ESCENARIO DE DESBALANCE SEVERO (Clase 'sci.med' = 5% del Corpus) ===")
print(f"MultinomialNB -> F1-Macro Global: {f1_macro_mnb:.4f} | F1 en Clase Minoritaria ('sci.med'): {f1_minoritaria_mnb:.4f}")
print(f"ComplementNB  -> F1-Macro Global: {f1_macro_cnb:.4f} | F1 en Clase Minoritaria ('sci.med'): {f1_minoritaria_cnb:.4f}")

In [ ]:
# 2. Pipeline de NLP y Optimización de Hiperparámetros con GridSearchCV (CountVectorizer vs TfidfVectorizer)
textos_full, y_full = generar_corpus_newsgroups([450, 450, 450, 450], seed=101)
X_train_txt, X_test_txt, y_train_txt, y_test_txt = train_test_split(
    textos_full, y_full, test_size=0.25, stratify=y_full, random_state=42
)

pipeline_nlp = Pipeline([
    ('vect', CountVectorizer()),
    ('clf', MultinomialNB())
])

param_grid = [
    {
        'vect': [CountVectorizer(), TfidfVectorizer()],
        'vect__ngram_range': [(1, 1), (1, 2)],
        'clf': [MultinomialNB(), ComplementNB(), BernoulliNB()],
        'clf__alpha': [0.01, 0.1, 0.5, 1.0, 2.0]
    }
]

cv_nlp = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_nlp = GridSearchCV(
    pipeline_nlp, param_grid=param_grid, cv=cv_nlp,
    scoring='f1_macro', n_jobs=-1, return_train_score=True
)
grid_nlp.fit(X_train_txt, y_train_txt)

print("=== MEJOR CONFIGURACIÓN ENCONTRADA POR GRIDSEARCHCV ===")
print(f"Mejores hiperparámetros: {grid_nlp.best_params_}")
print(f"Mejor F1-Macro (CV):     {grid_nlp.best_score_:.4f}")

y_pred_best = grid_nlp.predict(X_test_txt)
print(f"Exactitud en Test:       {accuracy_score(y_test_txt, y_pred_best):.4f}\n")
print(classification_report(y_test_txt, y_pred_best, target_names=categorias_nlp))

In [ ]:
# 3. Visualización: Matriz de Confusión y Top Palabras con Mayor Poder Discriminativo por Clase
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Panel Izquierdo: Matriz de Confusión
cm = confusion_matrix(y_test_txt, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=categorias_nlp)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Matriz de Confusión en Conjunto de Prueba (Test)', fontsize=12, fontweight='bold')
axes[0].grid(False)

# Panel Derecho: Sensibilidad a alpha según Clasificador (con TfidfVectorizer)
res_df = pd.DataFrame(grid_nlp.cv_results_)
mask_tfidf_uni = (
    res_df['param_vect'].astype(str).str.contains('TfidfVectorizer') &
    (res_df['param_vect__ngram_range'] == (1, 1))
)
df_plot = res_df[mask_tfidf_uni].copy()
df_plot['Modelo'] = df_plot['param_clf'].apply(lambda x: x.__class__.__name__)
df_plot['alpha'] = df_plot['param_clf__alpha'].astype(float)

for modelo, grp in df_plot.groupby('Modelo'):
    axes[1].plot(grp['alpha'], grp['mean_test_score'], marker='o', lw=2.3, label=modelo)

axes[1].set_xscale('log')
axes[1].set_xlabel(r'Parámetro de Suavizado $\alpha$ (Escala Log)', fontsize=11)
axes[1].set_ylabel('F1-Macro Promedio (5-Fold CV)', fontsize=11)
axes[1].set_title(r'Curva de Validación de $\alpha$ por Variante Bayesiana (TF-IDF)', fontsize=12, fontweight='bold')
axes[1].legend(frameon=True, facecolor='white', fontsize=10.5)

plt.tight_layout()
plt.show()

# Extracción de los Términos con Mayor Log-Probabilidad Condicional por Categoría
vec_interpretable = TfidfVectorizer(ngram_range=(1, 1)).fit(X_train_txt)
mnb_interpretable = MultinomialNB(alpha=0.5).fit(vec_interpretable.transform(X_train_txt), y_train_txt)
feature_names_nlp = np.array(vec_interpretable.get_feature_names_out())

print("=== TOP 8 TÉRMINOS CON MAYOR PODER DISCRIMINATIVO (ln P(w | Y=c_k)) POR CLASE ===")
for c_idx, cat_name in enumerate(categorias_nlp):
    top_indices = np.argsort(mnb_interpretable.feature_log_prob_[c_idx])[-8:][::-1]
    top_terms = [f"{feature_names_nlp[i]} ({mnb_interpretable.feature_log_prob_[c_idx, i]:.2f})" for i in top_indices]
    print(f"• {cat_name:<15}: {', '.join(top_terms)}")

---

## 7. Conclusiones y Reglas de Ingeniería para Modelos Discretos

1. **Elección de Variante en NLP:**
   * Si los documentos son cortos (tweets, asuntos de correo, SMS) y la no aparición de palabras clave aporta señal semántica, utiliza **`BernoulliNB`**.
   * Si el dataset presenta desbalance de clases importante, **`ComplementNB`** supera sistemáticamente a `MultinomialNB` en *Recall* y *F1-Macro* sobre las clases minoritarias.
   * Aunque `MultinomialNB` se deduce teóricamente sobre conteos enteros (`CountVectorizer`), en la práctica industrial funciona aún mejor con matrices de pesos fraccionarios **`TfidfVectorizer`** combinadas con un suavizado de Lidstone $\alpha \in [0.05, 0.5]$.
2. **Eficiencia con Matrices Dispersas (*Sparse Matrices*):**
   Jamás conviertas la salida de `CountVectorizer` o `TfidfVectorizer` a un arreglo denso con `.toarray()`. Los clasificadores `MultinomialNB`, `ComplementNB` y `BernoulliNB` operan nativamente sobre matrices comprimidas por filas (`scipy.sparse.csr_matrix`), permitiendo entrenar sobre vocabularios de $V > 100,000$ palabras usando apenas unos pocos megabytes de RAM.
3. **Variables Categóricas Tabulares:**
   Usa siempre `OrdinalEncoder` + `CategoricalNB(min_categories=...)` para variables categóricas nominales. Si un dataset tabular es **mixto** (combina columnas continuas y columnas categóricas), puedes sumar las log-verosimilitudes de un `GaussianNB` (sobre las continuas) y un `CategoricalNB` (sobre las categóricas) restando una vez el log-prior $\ln P(Y=c_k)$.

---
**Siguiente paso:** En el **Notebook 04 (`04_Calibracion_Streaming_y_Benchmark_Integral.ipynb`)** abordaremos la calibración probabilística (Platt Scaling vs Regresión Isotónica), el entrenamiento fuera de núcleo (*Out-of-Core Streaming*) con `partial_fit` y un benchmark competitivo contra Regresión Logística, KNN, Árboles CART y Random Forest.